# BGE-M3 -> UMAP -> HDBSCAN on Colab (Sri Lankan Hansard paper, then Conversation)
Code: GitHub `TriDang2025/attribute-extraction` (`hdbscan/`). Data: Hugging Face + the paper's public sources. Results: committed back to GitHub (`hdbscan/results/`).

**Before running:** Runtime > Change runtime type > a GPU (T4 is enough; L4 is faster).

In [ ]:
#@title 1. Tokens, clone/pull, git identity (run every session)
# Needs Colab Secrets (key icon on the left): HF_TOKEN and GH_TOKEN, both with notebook access on.
GIT_NAME = "Your Name"  #@param {type:"string"}
GIT_EMAIL = "you@users.noreply.github.com"  #@param {type:"string"}
import os, subprocess
from google.colab import userdata
from huggingface_hub import login

# Strip whitespace/newlines: a pasted secret often ends with one
hf_token = (userdata.get("HF_TOKEN") or "").strip()
if hf_token:
    login(token=hf_token)  # hf CLI + huggingface_hub, also in the Colab terminal
del hf_token
# GitHub token via an environment variable that every later cell and background job inherits.
# (git's credential *cache* runs as a helper process that Colab kills when the cell ends.)
os.environ["GH_TOKEN"] = (userdata.get("GH_TOKEN") or "").strip()
def run(cmd, **kw): subprocess.run(cmd, check=True, **kw)
run(["git", "config", "--global", "user.name", GIT_NAME])
run(["git", "config", "--global", "user.email", GIT_EMAIL])
run(["git", "config", "--global", "credential.helper",
     '!f() { echo username=token; echo "password=$GH_TOKEN"; }; f'])
REPO = "/content/attribute-extraction"
if not os.path.exists(REPO):
    run(["git", "clone", "-q", "https://github.com/TriDang2025/attribute-extraction.git", REPO])
else:
    run(["git", "-C", REPO, "pull", "-q", "--rebase", "--autostash"])
%cd /content/attribute-extraction/hdbscan
!git log -1 --format="code at %h (%an, %ar): %s"


## 2. Run everything in the background (about 1-2 h; resumes after a disconnect)
Embeddings are uploaded to HF; result tables are committed to GitHub at the end.

In [ ]:
!bash colab/run_colab.sh

## 3. Progress (rerun any time; ends with `HBE_DONE`)

In [ ]:
!bash colab/run_colab.sh status

## Optional: let Claude Code run it
Run the next cell, then open **Terminal** (bottom-left) and type:
```
export PATH=$HOME/.local/bin:$PATH
cd /content/attribute-extraction/hdbscan
claude
```
Log in when asked (open the link, sign in, paste the code back), then type:
`Read colab/COLAB_AGENT.md and follow it.`
Cell 1 must have run first (it sets up the Hugging Face and GitHub logins the agent uses).

In [ ]:
!curl -fsSL https://claude.ai/install.sh | bash